# 1. Import et nettoyage 

In [32]:
import hashlib, urllib.request
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

URL = "https://docs.ambroise.cloud/velib/velib-2026-10-08.csv.gz"
FILE = "velib-2026-10-08.csv.gz"
MD5 = "02ab1e3fdac10579e4ce93d1540a5dc7"

# 1. Download to disk, then verify the checksum
path, _ = urllib.request.urlretrieve(URL, FILE)
print("Download successful:", path)

print("------------------------------------------------------")

# 2. Verify MD5
with open(path, "rb") as f:
    calculated_md5 = hashlib.md5(f.read()).hexdigest()

print("Calculated MD5:", calculated_md5)
print("Expected MD5:  ", MD5)

assert calculated_md5 == MD5, f"MD5 mismatch: {calculated_md5}"
print("MD5 OK!")


Download successful: velib-2026-10-08.csv.gz
------------------------------------------------------
Calculated MD5: 02ab1e3fdac10579e4ce93d1540a5dc7
Expected MD5:   02ab1e3fdac10579e4ce93d1540a5dc7
MD5 OK!


In [36]:
# 3. Read, then convert the time columns
raw = pd.read_csv(path)
print("\n Dataset shape:", raw.shape)
print("   Number of rows:", raw.shape[0])
print("   Number of columns:", raw.shape[1])

print("------------------------------------------------------")
print("\n Column names:")
print(raw.columns.tolist())


 Dataset shape: (329680, 16)
   Number of rows: 329680
   Number of columns: 16
------------------------------------------------------

 Column names:
['collected_at_utc', 'source_updated_at', 'station_code', 'name', 'commune', 'insee_code', 'latitude', 'longitude', 'capacity', 'bikes_available', 'mechanical_bikes', 'electric_bikes', 'docks_available', 'is_installed', 'is_renting', 'is_returning']


In [37]:
# 4. Convert time columns
for c in ["collected_at_utc", "source_updated_at"]:
    raw[c] = pd.to_datetime(raw[c], utc=True)

df = raw.copy()   # `raw` stays untouched; all cleaning happens on df
print(df.head())
print(df.dtypes[["collected_at_utc", "source_updated_at"]])

           collected_at_utc         source_updated_at  station_code                             name commune  insee_code   latitude  longitude  capacity  bikes_available  mechanical_bikes  \
0 2026-08-25 11:57:26+00:00 2026-08-25 11:35:29+00:00         10017         Gare de l'Est - Fidélité   Paris       75056  48.874575   2.356796      15.0                8                 1   
1 2026-08-25 11:57:26+00:00 2026-08-25 11:34:30+00:00         10042  Enghien - Faubourg Poissonnière   Paris       75056  48.872420   2.348395      33.0               30                22   
2 2026-08-25 11:57:26+00:00 2026-08-25 11:32:01+00:00         10163         Gare du Nord - Demarquay   Paris       75056  48.881154   2.358411      30.0                2                 0   
3 2026-08-25 11:57:26+00:00 2026-08-25 11:32:46+00:00         10202       Bichat - Grange aux Belles   Paris       75056  48.873688   2.365015      30.0                4                 3   
4 2026-08-25 11:57:26+00:00 2026-08-25 11:33:

In [44]:
print(df.describe())

        station_code     insee_code       latitude      longitude       capacity  bikes_available  mechanical_bikes  electric_bikes  docks_available  is_installed     is_renting   is_returning
count  329680.000000  329680.000000  329680.000000  329680.000000  327760.000000    329680.000000     329680.000000   329680.000000    329680.000000      329680.0  329680.000000  329680.000000
mean    18749.075000   81699.862500      48.855573       2.331706      30.687500        12.920471          8.069695        4.850777        16.844352           1.0       0.997537       0.997537
std     11641.079442    8593.362126       0.033103       0.058007      12.438461        11.754529         10.053956        5.065029        12.120306           0.0       0.049567       0.049567
min      2102.000000   75056.000000      48.762770       2.188134      12.000000         0.000000          0.000000        0.000000         0.000000           1.0       0.000000       0.000000
25%      9791.750000   75056.000000

In [45]:
# Les valeurs dupplicants
print("\n Number of duplicate rows:", df.duplicated().sum())


 Number of duplicate rows: 0


In [47]:
# 8. Check missing values
print("\nMissing values per column:")
print(df.isna().sum())


Missing values per column:
collected_at_utc        0
source_updated_at       0
station_code            0
name                    0
commune                 0
insee_code              0
latitude                0
longitude               0
capacity             1920
bikes_available         0
mechanical_bikes        0
electric_bikes          0
docks_available         0
is_installed            0
is_renting              0
is_returning            0
dtype: int64


In [48]:
missing_capacity = df[df["capacity"].isna()]

print(missing_capacity.head())

               collected_at_utc         source_updated_at  station_code                               name          commune  insee_code   latitude  longitude  capacity  bikes_available  \
64370 2026-09-02 22:30:01+00:00 2026-09-02 21:59:19+00:00         31104          Mairie de Rosny-sous-Bois  Rosny-sous-Bois       93064  48.871257   2.486581       NaN               29   
64371 2026-09-02 22:30:01+00:00 2026-09-02 21:58:54+00:00         32001        Proudhon - Président Wilson      Saint-Denis       93066  48.906443   2.358491       NaN                9   
64372 2026-09-02 22:30:01+00:00 2026-09-02 21:56:16+00:00          3201       Payenne - Square George Caïn            Paris       75056  48.857833   2.362264       NaN               13   
64373 2026-09-02 22:30:01+00:00 2026-09-02 21:56:30+00:00         32020             Jules Guesdes - Carnot      Saint-Denis       93066  48.938282   2.350899       NaN               14   
64374 2026-09-02 22:30:01+00:00 2026-09-02 21:50:11+00:00   

In [ ]:
print(missing_capacity["name"].value_counts())

name
Mairie de Rosny-sous-Bois                24
Proudhon - Président Wilson              24
Payenne - Square George Caïn             24
Jules Guesdes - Carnot                   24
Square du Théâtre du Garde-Chasse        24
                                         ..
Labouret - Saint-Denis                   24
Richard Wallace - Place du 8 Mai 1945    24
Saint-Gilles - Turenne                   24
Archives - Pastourelle                   24
Lagny - Auguste Blanqui                  24
Name: count, Length: 80, dtype: int64


In [49]:
print("stations:", df.iloc[:, 0].nunique())
print("ange:", df["collected_at_utc"].min(), "→", df["collected_at_utc"].max())
print("snapshots:", df["collected_at_utc"].nunique())


stations: 4139
ange: 2026-08-25 11:57:26+00:00 → 2026-10-08 20:15:01+00:00
snapshots: 4139


In [50]:
print("rows:", len(df))                                       # 329 680
print("stations:", df["station_code"].nunique())              # 80
print("range:", df["collected_at_utc"].min(), "→", df["collected_at_utc"].max())
print("snapshots:", df["collected_at_utc"].nunique())         # 4 121
print("rows per station:", len(df) / df["station_code"].nunique())  # 4 121.0

rows: 329680
stations: 80
range: 2026-08-25 11:57:26+00:00 → 2026-10-08 20:15:01+00:00
snapshots: 4139
rows per station: 4121.0


In [51]:
print(df.groupby("station_code")["capacity"].nunique().value_counts())

capacity
1    80
Name: count, dtype: int64


In [52]:
miss = df[df["capacity"].isna()]
print(miss["collected_at_utc"].agg(["min", "max"]))

min   2026-09-02 22:30:01+00:00
max   2026-09-26 03:00:01+00:00
Name: collected_at_utc, dtype: datetime64[us, UTC]


In [54]:
print("range:", df["collected_at_utc"].min(), "->", df["collected_at_utc"].max())
print("snapshots:", df["collected_at_utc"].nunique())

range: 2026-08-25 11:57:26+00:00 -> 2026-10-08 20:15:01+00:00
snapshots: 4139
